In [10]:
import pandas as pd
import json
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Define file paths
passage_collection_path = "../data/raw/collection.tsv"
query_relevance_dev_path = "../data/raw/qrels_dev.tsv"
query_dev_path = "../data/raw/queries_dev.tsv"

# Read in the dataframes
passage_collection_df = pd.read_csv(passage_collection_path, sep="\t", names=["passage_id", "passage_text"])
query_relevance_dev_df = pd.read_csv(query_relevance_dev_path, sep="\t", names=["query_id", "query_relevance", "passage_id", "passage_relevance"])
query_dev_df = pd.read_csv(query_dev_path, sep="\t", names=["query_id", "query_text"])

In [3]:
# EDA
print("Passage Collection DataFrame Info:")
print(passage_collection_df.info())
print(passage_collection_df.head())

Passage Collection DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8841823 entries, 0 to 8841822
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   passage_id    int64 
 1   passage_text  object
dtypes: int64(1), object(1)
memory usage: 134.9+ MB
None
   passage_id                                       passage_text
0           0  The presence of communication amid scientific ...
1           1  The Manhattan Project and its atomic bomb help...
2           2  Essay on The Manhattan Project - The Manhattan...
3           3  The Manhattan Project was the name for a proje...
4           4  versions of each volume as well as complementa...


In [4]:
print("Query Relevance Dev DataFrame Info:")
print(query_relevance_dev_df.info())
print(query_relevance_dev_df.head())

Query Relevance Dev DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59273 entries, 0 to 59272
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   query_id           59273 non-null  int64
 1   query_relevance    59273 non-null  int64
 2   passage_id         59273 non-null  int64
 3   passage_relevance  59273 non-null  int64
dtypes: int64(4)
memory usage: 1.8 MB
None
   query_id  query_relevance  passage_id  passage_relevance
0   1102432                0     2026790                  1
1   1102431                0     7066866                  1
2   1102431                0     7066867                  1
3   1090282                0     7066900                  1
4     39449                0     7066905                  1


In [5]:
print("Query Dev DataFrame Info:")
print(query_dev_df.info())
print(query_dev_df.head())

Query Dev DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101093 entries, 0 to 101092
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   query_id    101093 non-null  int64 
 1   query_text  101093 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.5+ MB
None
   query_id                      query_text
0   1048578  cost of endless pools/swim spa
1   1048579                    what is pcnt
2   1048580               what is pcb waste
3   1048581                   what is pbis?
4   1048582                  what is paysky


In [6]:

# Check the if the `query_id` in the relevance dataframes are unique (i.e. if the len of unique values is the same as the len of the dataframe)
# If they are not unique, means that there are multiple passages relevant to the same query
qrel_dev_unique_queries = query_relevance_dev_df["query_id"].nunique()
qrel_dev_total_queries = len(query_relevance_dev_df)
print(f"\nQuery Relevance Dev DataFrame - Unique Queries: {qrel_dev_unique_queries}, Total Queries: {qrel_dev_total_queries}")


Query Relevance Dev DataFrame - Unique Queries: 55578, Total Queries: 59273


In [7]:
# Check for queries in dev set that are not in relevance dev set
dev_query_ids = set(query_dev_df["query_id"].unique())
relevance_dev_query_ids = set(query_relevance_dev_df["query_id"].unique())
missing_in_relevance_dev = dev_query_ids - relevance_dev_query_ids
print(f"Number of query IDs in dev set not in relevance dev set: {len(missing_in_relevance_dev)}")

Number of query IDs in dev set not in relevance dev set: 45515


In [8]:
# Check for passages that are in relevance dataframes but not in passage collection
passage_ids_in_collection = set(passage_collection_df["passage_id"].unique())
passage_ids_in_relevance_dev = set(query_relevance_dev_df["passage_id"].unique())
missing_passages_in_collection_dev = passage_ids_in_relevance_dev - passage_ids_in_collection
print(f"Number of passage IDs in relevance dev set not in passage collection: {len(missing_passages_in_collection_dev)}")

Number of passage IDs in relevance dev set not in passage collection: 0


In [9]:
# Filter for queries in dev set that are in relevance dev set
filtered_query_dev_df = query_dev_df[query_dev_df["query_id"].isin(relevance_dev_query_ids)]
print(f"Filtered Query Dev DataFrame Length (only queries present in relevance dev set):")
print(len(filtered_query_dev_df))

Filtered Query Dev DataFrame Length (only queries present in relevance dev set):
55578


In [11]:
# Split the filtered query dev dataframe into train and validation sets (60-20-20 split)
train_df, temp_df = train_test_split(filtered_query_dev_df, test_size=0.4, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"Train DataFrame Length: {len(train_df)}")
print(f"Validation DataFrame Length: {len(val_df)}")
print(f"Test DataFrame Length: {len(test_df)}")

Train DataFrame Length: 33346
Validation DataFrame Length: 11116
Test DataFrame Length: 11116


In [12]:
# Create a mapping of query to relevant passages for fast lookup
query_relevance_dev_dict = (query_relevance_dev_df.groupby("query_id")["passage_id"]
                            .apply(lambda x: [str(pid) for pid in x])
                            .to_dict())

# For each query in dev set, we want to format it as:
# {
#   "qid": "1234",
#   "query": "how to change iphone battery",
#   "qrels": ["201", "55689"]  // from qrels.dev
# }

formatted_train_data_output_path = "../data/processed/formatted_train_data.jsonl"
formatted_val_data_output_path = "../data/processed/formatted_val_data.jsonl"
formatted_test_data_output_path = "../data/processed/formatted_test_data.jsonl"

formatted_train_data = []
formatted_val_data = []
formatted_test_data = []

for _, row in train_df.iterrows():
    qid = row["query_id"]
    query_text = row["query_text"]
    qrels = query_relevance_dev_dict.get(qid, [])
    formatted_train_data.append({
        "qid": str(qid),
        "query": query_text,
        "qrels": qrels
    })

for _, row in val_df.iterrows():
    qid = row["query_id"]
    query_text = row["query_text"]
    qrels = query_relevance_dev_dict.get(qid, [])
    formatted_val_data.append({
        "qid": str(qid),
        "query": query_text,
        "qrels": qrels
    })

for _, row in test_df.iterrows():
    qid = row["query_id"]
    query_text = row["query_text"]
    qrels = query_relevance_dev_dict.get(qid, [])
    formatted_test_data.append({
        "qid": str(qid),
        "query": query_text,
        "qrels": qrels
    })

# Save formatted data to be JSONL
with open(formatted_train_data_output_path, "w") as f:
    for item in formatted_train_data:
        f.write(json.dumps(item) + "\n")

with open(formatted_val_data_output_path, "w") as f:
    for item in formatted_val_data:
        f.write(json.dumps(item) + "\n")

with open(formatted_test_data_output_path, "w") as f:
    for item in formatted_test_data:
        f.write(json.dumps(item) + "\n")

In [15]:
# Get the set of passage IDs that are not present in dev relevance dataframe
all_passage_ids = set(passage_collection_df["passage_id"].unique())
relevant_passage_ids = passage_ids_in_relevance_dev
non_relevant_passage_ids = all_passage_ids - passage_ids_in_relevance_dev
print(f"Total unique relevant passage IDs: {len(relevant_passage_ids)}")
print(f"Total unique non-relevant passage IDs: {len(non_relevant_passage_ids)}")

Total unique relevant passage IDs: 59096
Total unique non-relevant passage IDs: 8782727


In [16]:
# Randomly sample 50,000 non-relevant passage IDs
# Set seed to be able to reproduce results
np.random.seed(42)
sampled_non_relevant_passage_ids = set(
    np.random.choice(list(non_relevant_passage_ids), size=50000, replace=False)
)

# Create a union of relevant passage IDs and sampled non-relevant passage IDs
final_passage_ids = relevant_passage_ids.union(sampled_non_relevant_passage_ids)
print(f"Total passage IDs after combining relevant and sampled non-relevant: {len(final_passage_ids)}")


Total passage IDs after combining relevant and sampled non-relevant: 109096


In [17]:
# Create a filtered passage collection dataframe with only relevant and sampled non-relevant passages
filtered_passage_collection_df = passage_collection_df[
    passage_collection_df["passage_id"].isin(final_passage_ids)
]

# Save the filtered passage collection as a JSONL file with each line as {"pid": ..., "passage": ...}
filtered_passage_collection_output_path = "../data/processed/filtered_passage_collection.jsonl"

with open(filtered_passage_collection_output_path, "w") as f:
    for _, row in filtered_passage_collection_df.iterrows():
        record = {
            "pid": str(row["passage_id"]),
            "passage": row["passage_text"]
        }
        f.write(json.dumps(record) + "\n")